In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!unzip -q -o /content/Adaora.zip -d /content
!ls /content/Adaora

In [11]:
import os
IMG = "/content/drive/MyDrive/capstone_clearance_nbi/images"
print(len(os.listdir(IMG)))                       # expect about 80
print(os.listdir(IMG + "/nbi_016261000000000"))   # expect 4 .jpg files

80
['01_1231711027273529.jpg', '00_514537106309052.jpg', '02_560075165425687.jpg', '03_4318414618246493.jpg', '05_1231711027273529.jpg', '00_268160684815657.jpg', '01_1787167231477236.jpg', '02_547477103189170.jpg', '03_978712926009248.jpg', '04_514537106309052.jpg', '06_560075165425687.jpg', '07_4318414618246493.jpg']


In [ ]:
!pip -q install transformers scipy pandas plotly einops safetensors
!pip -q install git+https://github.com/facebookresearch/vggt.git

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tifffile 2026.10.15 requires numpy>=2.1, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.11.1 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jax 0.11.1 requires numpy>=2.1, but you have numpy 1.26.4 which is incompatible.
ml-dtypes 0.6.0 requires numpy>=2.0.0, but you have numpy 1.26.4 which is incompatible.
ml-dtypes 0.6.0 requires numpy>=2.1.0; python_version >= "3.13", but you have numpy 1.26.4 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1

In [9]:
# %% [markdown]
# # VGGT on the frozen 80 NBI bridge cases
# Same frames, segmenter, clearance metric and QC as MapAnything (`clearance_core.py`, unchanged).
# Only the 3D backbone differs: VGGT-1B, GPS-scaled (VGGT is not metric). Multi-view only.
# Run on Colab with an L4/A100 GPU. Resumable: one JSON per case in CASE_DIR.

import os, sys, json, ast, time, math
from pathlib import Path
import numpy as np, pandas as pd, torch
from PIL import Image

PACK     = Path("/content/Adaora")
IMG_ROOT = Path("/content/drive/MyDrive/capstone_clearance_nbi/images")
OUT_DIR  = Path("/content/drive/MyDrive/adaora_vggt")
CASE_DIR  = OUT_DIR / "cases"; CASE_DIR.mkdir(parents=True, exist_ok=True)
FORCE_RERUN = False
RESULT_VERSION = "vggt_gps_v1"
MIN_GPS_BASELINE_M = 3.0
VGGT_W = 518                                                # VGGT native width; height keeps aspect, multiple of 14

sys.path.insert(0, str(PACK / "reference"))
from clearance_core import *          # loads Mask2Former; defines sam3_segment, road_union, select_structure, estimate_clearance, qc_pass
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

from vggt.models.vggt import VGGT
from vggt.utils.pose_enc import pose_encoding_to_extri_intri
from vggt.utils.geometry import unproject_depth_map_to_point_map
vggt = VGGT.from_pretrained("facebook/VGGT-1B").to(DEVICE).eval()
DTYPE = torch.bfloat16 if (DEVICE == "cuda" and torch.cuda.get_device_capability()[0] >= 8) else torch.float16

# %% Cases and frame lookup
cl = pd.read_csv(PACK / "data/case_list_frozen_80.csv")
meta = {c["case_id"]: c for c in json.load(open(PACK / "data/cases.json"))}
assert len(cl) == 80

def frame_path(case_id, image_id):
    hits = sorted((IMG_ROOT / case_id).glob(f"*_{image_id}.jpg"))
    if not hits: raise FileNotFoundError(f"{case_id}/{image_id}")
    return hits[0]

def haversine_m(lat1, lon1, lat2, lon2):
    R = 6371008.8
    p1, p2 = math.radians(lat1), math.radians(lat2)
    a = math.sin((p2 - p1) / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(math.radians(lon2 - lon1) / 2) ** 2
    return 2 * R * math.asin(math.sqrt(a))

# %% VGGT -> metric `ma`
def load_for_vggt(imgs, W=VGGT_W):
    """Resize (no crop, so masks stay aligned with the full image) to W x H, H from the anchor's aspect, multiple of 14."""
    h0 = imgs[0].height * W / imgs[0].width
    H = int(min(924, max(14, round(h0 / 14) * 14)))
    arr = [torch.from_numpy(np.asarray(im.resize((W, H), Image.BICUBIC))).permute(2, 0, 1).float() / 255 for im in imgs]
    return torch.stack(arr)                                  # (S,3,H,W)

def run_vggt(imgs):
    x = load_for_vggt(imgs).to(DEVICE)
    with torch.inference_mode(), torch.autocast(DEVICE, dtype=DTYPE, enabled=DEVICE == "cuda"):
        pred = vggt(x[None])
    extri, intri = pose_encoding_to_extri_intri(pred["pose_enc"], x.shape[-2:])
    ext = extri[0].float().cpu().numpy()                       # (S,3,4) world->cam (OpenCV)
    K = intri[0].float().cpu().numpy()
    depth = pred["depth"][0].float().cpu().numpy()             # (S,H,W,1)
    conf = pred["depth_conf"][0].float().cpu().numpy()         # (S,H,W)
    pts = unproject_depth_map_to_point_map(depth, ext, K)      # (S,H,W,3) in the world frame of frame 0
    R, t = ext[:, :, :3], ext[:, :, 3]
    centres = -np.einsum("sji,sj->si", R, t)                   # c = -R^T t
    del pred, x
    return {"pts": pts.astype(np.float32), "conf": conf.astype(np.float32), "centres": centres}

def gps_scale(centres, lat_lon):
    """s = median over frame pairs (GPS distance / VGGT camera-centre distance); pairs with GPS baseline >= 3 m only."""
    ratios = []
    for i in range(len(centres)):
        for j in range(i + 1, len(centres)):
            g = haversine_m(*lat_lon[i], *lat_lon[j]); v = float(np.linalg.norm(centres[i] - centres[j]))
            if g >= MIN_GPS_BASELINE_M and v > 1e-6:
                ratios.append(g / v)
    return (float(np.median(ratios)) if ratios else None), len(ratios), ratios

def build_ma(v, k, s):
    pts, conf = v["pts"][k] * s, v["conf"][k]
    finite = np.isfinite(pts).all(-1) & np.isfinite(conf)
    strict = finite & (conf >= np.percentile(conf[finite], 50))
    loose = finite & (conf >= np.percentile(conf[finite], 15))
    return {"pts3d": pts, "conf": conf, "strict": strict, "loose": loose, "cam": (v["centres"][k] * s).astype(np.float64)}

# %% Per-case processing
def process_case(row):
    cid = row.case_id; res_path = CASE_DIR / f"{cid}.json"
    if res_path.exists() and not FORCE_RERUN:
        c = json.loads(res_path.read_text())
        if c.get("version") == RESULT_VERSION: return c
    t0 = time.time()
    view_ids = [str(v) for v in ast.literal_eval(row.view_image_ids)]
    anchor_id = str(row.anchor_image_id); ai = view_ids.index(anchor_id)
    rec = {"version": RESULT_VERSION, "case_id": cid, "gt_clearance_m": float(row.gt_clearance_m), "anchor_image_id": anchor_id,
           "n_views": len(view_ids), "cam_make": row.cam_make, "exif_camera": bool(row.exif_camera), "view_mode": "multi"}
    imgs = [Image.open(frame_path(cid, i)).convert("RGB") for i in view_ids]
    sd, rd = sam3_segment(imgs[ai])
    road_m, _ = road_union(rd, (imgs[ai].height, imgs[ai].width))
    st = select_structure(sd, road_m, (imgs[ai].height, imgs[ai].width))
    if st is None or road_m is None:
        rec.update(status="no_structure_on_anchor" if st is None else "no_road_on_anchor", qc_pass=False)
    else:
        fm = {f["image_id"]: f for f in meta[cid]["frames"]}
        ll = [(fm[i]["lat"], fm[i]["lon"]) for i in view_ids]
        v = run_vggt(imgs)
        s, n_pairs, ratios = gps_scale(v["centres"], ll)
        rec.update(gps_pairs_valid=n_pairs, scale_s=s, scale_s_min=min(ratios) if ratios else None, scale_s_max=max(ratios) if ratios else None)
        if s is None:
            rec.update(status="no_gps_baseline", qc_pass=False)
        else:
            est = estimate_clearance(build_ma(v, ai, s), st["mask"], road_m)
            if est.get("ok"):
                rec.update(status="ok", reason="", clear_med_m=est["clear_med_m"], clear_p10_m=est["clear_p10_m"], cam_height_m=est["cam_height_m"],
                           n_cols=est["n_cols"], struct_dist_m=est["struct_dist_m"], qc_pass=qc_pass(est))
            else:
                rec.update(status="estimate_failed", reason=est.get("reason"), qc_pass=False)
        del v
    rec["runtime_s"] = round(time.time() - t0, 1)
    res_path.write_text(json.dumps(rec, indent=1, default=str)); return rec

recs = []
for r in cl.itertuples():
    try: recs.append(process_case(r))
    except Exception as e:
        recs.append({"case_id": r.case_id, "status": "error", "reason": repr(e)[:200], "qc_pass": False}); print("ERR", r.case_id, e)
    if DEVICE == "cuda": torch.cuda.empty_cache()

# %% Write results in tools/results_template.csv format (+ scale diagnostics in a separate CSV)
cols = ["method","view_mode","case_id","gt_clearance_m","anchor_image_id","n_views","status","reason","clear_med_m","clear_p10_m",
        "cam_height_m","n_cols","struct_dist_m","qc_pass","cam_make","exif_camera","runtime_s"]
df = pd.DataFrame(recs).merge(cl[["case_id","gt_clearance_m","cam_make","exif_camera","anchor_image_id","n_views"]], on="case_id", how="left", suffixes=("_x",""))
df = df[[c for c in df.columns if not c.endswith("_x")]]
df["method"], df["view_mode"] = "VGGT", "multi"
for c in cols:
    if c not in df: df[c] = np.nan
df["qc_pass"] = df["qc_pass"].fillna(False).astype(bool)
df[cols].to_csv(OUT_DIR / "adaora_vggt_results.csv", index=False)
df[["case_id","status","gps_pairs_valid","scale_s","scale_s_min","scale_s_max","cam_height_m"]].to_csv(OUT_DIR / "adaora_vggt_scale_diagnostics.csv", index=False)
print(df.status.value_counts())
# then:  !python /content/Adaora/tools/metrics.py /content/drive/MyDrive/adaora_vggt/adaora_vggt_results.csv


ModuleNotFoundError: No module named 'clearance_core'